In [ ]:
import json
import os
from urllib.parse import quote as url_quote

from requests import delete, get, post

# Generic utils
from utils import setup_notebook

# Check environment setup
if not setup_notebook(
    required_keys=[
        "GROQ_API_KEY",
    ]
):
    raise OSError("Please set up required environment variables")

In [ ]:
NAME = "Test Agent Groq: examples/create-agent-groq.ipynb"

response = post(
    "http://localhost:8000/api/v2/agents",
    json={
        "mode": "conversational",
        "name": NAME,
        "version": "1.0.0",
        "description": (
            "This is a test agent for Groq.\nThis agent uses the Groq PlatformClient."
        ),
        "runbook": "# Objective\nYou are a helpful assistant.",
        "platform_configs": [
            {
                "kind": "groq",
                "groq_api_key": os.environ["GROQ_API_KEY"],
            }
        ],
        "action_packages": [
            {
                "name": "serp-api",
                "version": "1.0.1",
                "organization": "testorg",
                "url": "localhost:8083",
                "api_key": {"value": "testing"},
                "allowed_actions": [],
            }
        ],
        "mcp_servers": [
            {
                "name": "browserbase-mcp-server",
                "url": "http://localhost:3001/sse",
            }
        ],
        "agent_architecture": {
            "name": "agent_platform.architectures.default",
            "version": "1.0.0",
        },
        "observability_configs": [],
        "question_groups": [],
        "extra": {
            "test": "test",
        },
    },
)
print(response.json())

In [ ]:
name_url_encoded = url_quote(NAME)
agent_by_name = get(
    f"http://localhost:8000/api/v2/agents/by-name?name={name_url_encoded}"
).json()

# Get id to use in the next few examples
agent_id = agent_by_name["agent_id"]

In [ ]:
from widget import DebugChatWidget

widget = DebugChatWidget(
    agent_id=agent_id,
    base_url="http://localhost:8000/api/v2",
)
widget

In [ ]:
# Now list threads for the agent
specific_thread = get(
    f"http://localhost:8000/api/v2/threads?agent_id={agent_id}"
).json()
print(json.dumps(specific_thread, indent=2))

In [ ]:
specific_agent = get(f"http://localhost:8000/api/v2/agents/{agent_id}").json()
print(json.dumps(specific_agent, indent=2))

In [ ]:
delete(f"http://localhost:8000/api/v2/agents/{agent_id}")

In [ ]:
# If we get the agent again, it should be gone
specific_agent = get(f"http://localhost:8000/api/v2/agents/{agent_id}")
print(
    json.dumps(
        {
            "status_code": specific_agent.status_code,
            "response": specific_agent.json(),
        },
        indent=2,
    )
)